<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/NLP/K%C3%A9rd%C3%A9s%20V%C3%A1lasz%20Rendszerek%20El%C5%91tan%C3%ADtott%20Modellekkel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kérdés-Válasz Rendszerek Előtanított Modellekkel

**Licenc: CC BY-NC-SA 4.0**

A **kérdés-válasz** (Question Answering, QA) rendszerek automatikusan megtalálják a választ egy kérdésre, akár strukturált szövegből, akár tudásbázisból.

## QA típusok

| Típus | Leírás | Bemenet | Kimenet |
|-------|--------|---------|---------|
| **Extractive** | Válasz kijelölése a kontextusból | Kérdés + Kontextus | Start-End pozíció |
| **Abstractive** | Válasz generálása | Kérdés + Kontextus | Generált szöveg |
| **Open-domain** | Nincs megadott kontextus | Kérdés | Tudásbázisból válasz |
| **Multiple-choice** | Válaszlehetőségekből választás | Kérdés + Opciók | Helyes opció |

## Példa (Extractive QA)

```
Kontextus: "William Shakespeare angol drámaíró és költő volt."
Kérdés:    "Mi volt Shakespeare foglalkozása?"
Válasz:    "drámaíró és költő" (start=20, end=36)
```

## Tartalomjegyzék
1. Extractive QA architektúra
2. Start-End pozíció predikció
3. Loss és metrikák
4. Benchmark datasetek

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Extractive QA architektúra

Az extractive QA a válasz **kezdő és végző pozícióját** jósolja meg a kontextusban.

### BERT-alapú QA modell

```
Bemenet:
[CLS] Ki volt Shakespeare? [SEP] William Shakespeare angol költő volt. [SEP]
  ↓         ↓                ↓              ↓              ↓         ↓
  0         1-4              5              6-7            8-9       10
        (Question)       (Separator)      (Answer)     (Context)
```

### Működés

1. **Tokenizálás**: Kérdés és kontextus összefűzése
2. **BERT encoding**: Kontextuális reprezentáció minden tokenre
3. **Start head**: Lineáris réteg a kezdő pozícióra
4. **End head**: Lineáris réteg a végző pozícióra
5. **Span extraction**: A legmagasabb score-ú start-end pár kiválasztása

### Constraint

A válasz span-nek **érvényesnek** kell lennie:
- `start <= end` (nem lehet negatív hossz)
- `end - start < max_answer_length` (túl hosszú válasz elkerülése)
- A válasz a **kontextusban** van (nem a kérdésben)

In [ ]:
class ExtractiveQA(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        # Encoder (egyszerűsített)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 4, batch_first=True),
            num_layers=2
        )
        self.embedding = nn.Embedding(10000, d_model)

        # Start és End pozíció predikció
        self.start_head = nn.Linear(d_model, 1)
        self.end_head = nn.Linear(d_model, 1)

    def forward(self, x):
        h = self.embedding(x)
        h = self.encoder(h)

        start_logits = self.start_head(h).squeeze(-1)  # [B, L]
        end_logits = self.end_head(h).squeeze(-1)

        return start_logits, end_logits

model = ExtractiveQA()
x = torch.randint(0, 10000, (2, 50))  # [question + context]
start, end = model(x)
print(f"Start logits: {start.shape}, End logits: {end.shape}")

In [ ]:
# Válasz kinyerése
def extract_answer(start_logits, end_logits, tokens, max_len=15):
    start_probs = F.softmax(start_logits, dim=-1)
    end_probs = F.softmax(end_logits, dim=-1)

    # Legjobb start-end pár keresése
    best_score = -float('inf')
    best_start, best_end = 0, 0

    for s in range(len(tokens)):
        for e in range(s, min(s + max_len, len(tokens))):
            score = start_probs[s] + end_probs[e]
            if score > best_score:
                best_score = score
                best_start, best_end = s, e

    return tokens[best_start:best_end+1]

# Vizualizáció
tokens = ['[CLS]', 'Ki', 'volt', 'Shakespeare', '?', '[SEP]',
          'William', 'Shakespeare', 'angol', 'költő', 'volt', '[SEP]']

# Szimulált logits
start_logits = torch.tensor([0, 0, 0, 0, 0, 0, 2.5, 0.5, 0, 0, 0, 0])
end_logits = torch.tensor([0, 0, 0, 0, 0, 0, 0.5, 2.8, 0, 0, 0, 0])

fig, ax = plt.subplots(figsize=(12, 3))
x_pos = np.arange(len(tokens))
ax.bar(x_pos - 0.15, F.softmax(start_logits, dim=0), 0.3, label='Start', color='green')
ax.bar(x_pos + 0.15, F.softmax(end_logits, dim=0), 0.3, label='End', color='red')
ax.set_xticks(x_pos)
ax.set_xticklabels(tokens, rotation=45)
ax.legend()
ax.set_title('Start/End pozíció valószínűségek')
plt.tight_layout()
plt.show()

answer = extract_answer(start_logits, end_logits, tokens)
print(f"Válasz: {' '.join(answer)}")

## 2. Hugging Face QA Pipeline

A HuggingFace `transformers` library egyszerűvé teszi a QA modellek használatát.

In [ ]:
print("""
from transformers import pipeline

qa = pipeline('question-answering', model='bert-large-uncased-whole-word-masking-finetuned-squad')

result = qa(
    question="Ki írta a Hamletet?",
    context="William Shakespeare angol drámaíró és költő volt. A Hamlet egyik leghíresebb műve."
)

print(result)
# {'answer': 'William Shakespeare', 'score': 0.98, 'start': 0, 'end': 19}
""")

## 3. Loss és metrikák

### Loss függvény

A QA loss a **start** és **end** pozíciók cross-entropy loss-ának összege:

$$\mathcal{L} = \frac{1}{2}(\text{CE}(p_{start}, y_{start}) + \text{CE}(p_{end}, y_{end}))$$

### Evaluation metrikák

| Metrika | Leírás | Számítás |
|---------|--------|----------|
| **Exact Match (EM)** | Tökéletes egyezés | 1 ha pred == gold, 0 egyébként |
| **F1 Score** | Token overlap | Precision × Recall / (P + R) |

### F1 vs EM

- **EM = 1**: "William Shakespeare" == "William Shakespeare"
- **EM = 0, F1 = 0.67**: "William Shakespeare" vs "Shakespeare" (1 közös / 1.5 avg)

In [ ]:
def qa_loss(start_logits, end_logits, start_pos, end_pos):
    start_loss = F.cross_entropy(start_logits, start_pos)
    end_loss = F.cross_entropy(end_logits, end_pos)
    return (start_loss + end_loss) / 2

def f1_score(pred_tokens, gold_tokens):
    pred_set = set(pred_tokens)
    gold_set = set(gold_tokens)

    if not pred_set or not gold_set:
        return 0.0

    overlap = pred_set & gold_set
    precision = len(overlap) / len(pred_set)
    recall = len(overlap) / len(gold_set)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

print(f"F1('William Shakespeare', 'Shakespeare'): {f1_score(['William', 'Shakespeare'], ['Shakespeare']):.2f}")

## 4. Benchmark datasetek

### Népszerű QA datasetek

| Dataset | Méret | Típus | Leírás |
|---------|-------|-------|--------|
| **SQuAD 1.1** | 100k+ | Extractive | Wikipedia alapú |
| **SQuAD 2.0** | 150k+ | + Unanswerable | Megválaszolhatatlan kérdések |
| **Natural Questions** | 300k+ | Real queries | Google keresések |
| **TriviaQA** | 650k+ | Trivia | Kvíz kérdések |
| **HotpotQA** | 113k | Multi-hop | Több dokumentum |

### SQuAD formátum

```json
{
  "context": "William Shakespeare angol drámaíró volt...",
  "question": "Mi volt Shakespeare foglalkozása?",
  "answers": {
    "text": ["drámaíró"],
    "answer_start": [20]
  }
}
```

## Összefoglalás

### Főbb tanulságok

1. **Extractive QA**: Start-end pozíció predikció a kontextusban
2. **BERT**: Kétirányú kontextus, ideális QA-hoz
3. **F1 és EM**: Standard metrikák, F1 engedékenyebb
4. **SQuAD**: A legfontosabb benchmark

### Kihívások

- **Unanswerable questions**: A modellnek tudnia kell, ha nincs válasz
- **Multi-hop reasoning**: Több szövegrészlet összekapcsolása
- **Long context**: 512+ token kontextus kezelése

### Következő lépések

A következő notebookban a **Large Language Models (LLMs)** világába lépünk, ahol a QA mellett generálás, reasoning és egyéb képességeket is látunk!